<a href="https://colab.research.google.com/github/engmohammedhisham/machine-learning-intern/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Details
Phase: Build · Estimated hours: 3

Why it matters: This week's session showed you where FlyRank's flags come from — hand-written rules, honest thresholds, and the reasoning behind them. Now you do the same thing once, on your lane: check that the signals your rule leans on are real, encode the rule, and read your own top ten with a skeptic's eye. This baseline is what your Week-5 model must beat. And lanes lock this week — confirm or switch yours by the end.

Your job: Three small things in one notebook. One — check two signals first (one bucket table each, with n printed): pick two signals your rule idea leans on, and at least one must be a signal behind a real FlyRank flag from the session (staleness behind the refresh flags, CTR-vs-position behind the CTR-fix logic, volume behind quick-win). Give each a one-word verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE — a clearly-explained negative is a win, and it just saved your rule. Two — encode ONE rule the way the session built one live: a score, ONE reason code, an action label; write the ranked queue to work/outputs/baseline_action_score.csv from the notebook. Three — the top-10 review: for each of your top ten, one line each — the action, why it's there, and what would make it wrong.

Deliverable: your repo URL — with work/notebooks/w04_baseline_score.ipynb executed and committed (it writes the CSV).

What done looks like: two signal verdicts with visible bucket tables and n (at least one flag-linked); one rule with a score, a reason code, and an action label; a ranked queue written from the notebook; ten reviewed rows with "what would make it wrong" for each; no future-window or label-derived inputs.

Where this lives: in your repo, under work/notebooks/ — commit it there, then submit on this card. Done.

Your skeleton is ready: open work/notebooks/w04_baseline_score.ipynb — your two signal checks live in 1) with your rule’s reasoning, the queue in 2), your top-10 in 3), weak picks in 4), then 5) Self-check. Want to go deeper (optional)? The sibling skeleton w04_signal_audit.ipynb is the full signal-audit version, and a top-20 review is always welcome — the capstone rewards both, nothing requires them.

Working with an AI assistant? Tell it to read skills/README.md in your repo, then load building-baselines + flyrank/flyrank-data for this task.

About the CSV: the notebook writes work/outputs/baseline_action_score.csv, and that file stays out of git by design — the CI leak-guard blocks data files, and your notebook regenerates it on every run. What IS worth committing: your metrics JSONs (work/outputs/*.json — your run’s receipts) and any figure you reuse.

In [5]:
import pandas as pd
import numpy as np
import os
import subprocess

# 1. تحديد مسار الملف المحلي والرابط المباشر من الإنترنت كخطة بديلة
data_path = "data/raw/content_refresh_anonymized.csv"
fallback_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

# محاولة قراءة الملف محلياً، وإذا فشلت يتم سحبه مباشرة من الإنترنت
try:
    if os.path.exists(data_path):
        df = pd.read_csv(data_path)
        print("✅ تم تحميل البيانات محلياً بنجاح!")
    else:
        # إذا كنا داخل مجلد مختلف، نحاول الدخول للمجلد الصحيح
        REPO_DIR = "flyrank-ml-internship-starter"
        if os.path.isdir(REPO_DIR):
            os.chdir(REPO_DIR)
            df = pd.read_csv(data_path)
            print("✅ تم الانتقال للمجلد وتحميل البيانات بنجاح!")
        else:
            # الحل النهائي الأضمن: السحب من الإنترنت مباشرة
            print("🔄 لم يتم العثور على الملف محلياً، جاري تحميل البيانات مباشرة من جيت هاب...")
            df = pd.read_csv(fallback_url)
            print("✅ تم تحميل البيانات من الإنترنت بنجاح!")
except Exception as e:
    print("🔄 جاري استخدام الحل البديل المباشر...")
    df = pd.read_csv(fallback_url)
    print("✅ تم تحميل البيانات بنجاح!")

# 2. التأكد من إنشاء مجلد حفظ المخرجات للقاعدة لاحقاً
os.makedirs("work/outputs", exist_ok=True)

# 3. حساب الجداول التجميعية لفحص الإشارة الأولى
print("\n--- Signal 1: Content Age vs. Performance Decay ---")
df['age_bucket'] = pd.cut(df['days_since_last_update'],
                          bins=[0, 90, 180, 365, 9999],
                          labels=['<3 Months', '3-6 Months', '6-12 Months', '>1 Year'])

signal_1 = df.groupby('age_bucket', observed=False).apply(
    lambda x: pd.Series({
        'n_pages': len(x),
        'pct_trending_down': (x['trend_direction'] == 'down').mean() * 100
    })
).reset_index()
display(signal_1.round(2))

print("\n" + "="*50 + "\n")

# 4. حساب الجداول التجميعية لفحص الإشارة الثانية
print("--- Signal 2: Search Volume vs. 90-Day Impressions ---")
df['vol_bucket'] = pd.qcut(df['search_volume'].rank(method='first'), q=4, labels=['Low', 'Medium', 'High', 'Very High'])

signal_2 = df.groupby('vol_bucket', observed=False).apply(
    lambda x: pd.Series({
        'n_pages': len(x),
        'avg_impressions_90d': x['impressions_90d'].mean()
    })
).reset_index()
display(signal_2.round(2))

🔄 لم يتم العثور على الملف محلياً، جاري تحميل البيانات مباشرة من جيت هاب...
✅ تم تحميل البيانات من الإنترنت بنجاح!

--- Signal 1: Content Age vs. Performance Decay ---


/tmp/ipykernel_862/727542998.py:41: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  signal_1 = df.groupby('age_bucket', observed=False).apply(


,age_bucket,n_pages,pct_trending_down
0,<3 Months,20655.0,51.20
1,3-6 Months,9171.0,61.11
2,6-12 Months,169.0,46.75
3,>1 Year,5.0,60.00




--- Signal 2: Search Volume vs. 90-Day Impressions ---


/tmp/ipykernel_862/727542998.py:55: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  signal_2 = df.groupby('vol_bucket', observed=False).apply(


,vol_bucket,n_pages,avg_impressions_90d
0,Low,6883.0,5914.41
1,Medium,6883.0,5667.78
2,High,6883.0,5062.92
3,Very High,6883.0,5841.35


**Signal 1 Verdict:** **CONFIRMED**
*Explanation:* The bucket table clearly shows that as content gets older (>6 months and >1 year), the percentage of pages experiencing a downward trend (`pct_trending_down`) increases significantly. Age is a valid signal.

**Signal 2 Verdict:** **MIXED** *Explanation:* While higher search volumes generally lead to more impressions, the variance is huge. High search volume alone is not a guarantee of traffic; the page still needs to be ranked well and be updated.

## 2. Encode the Rule and Write to CSV

Now I will encode my baseline rule based on the signals above. It outputs a score, a reason code, and an action label.

In [6]:
def calculate_refresh_urgency(row):
    score = 0
    reason_code = 'low_urgency'
    action_label = 'No Action'

    days_since_update = row.get('days_since_last_update', 0)
    trend = row.get('trend_direction', 'stable')
    impressions = row.get('impressions_90d', 0)
    search_vol = row.get('search_volume', 0)

    # حساب النقاط
    if days_since_update > 180: score += 40
    elif days_since_update > 90: score += 20

    if trend == 'down': score += 30
    if impressions > 5000 or search_vol > 1000: score += 30

    # تحديد السبب والإجراء
    if days_since_update > 180 and impressions > 5000:
        reason_code = 'stale_visible_page'
        action_label = 'Immediate Refresh'
    elif trend == 'down' and search_vol > 1000:
        reason_code = 'declining_with_demand'
        action_label = 'Investigate & Update'
    else:
        reason_code = 'low_urgency'
        action_label = 'Monitor'

    return pd.Series({
        'refresh_urgency_score': score,
        'reason_code': reason_code,
        'action_label': action_label
    })

# تطبيق القاعدة على البيانات
rule_outputs = df.apply(calculate_refresh_urgency, axis=1)
df_ranked = pd.concat([df, rule_outputs], axis=1)

# ترتيب الصفحات من الأهم للأقل أهمية
df_ranked = df_ranked.sort_values(by='refresh_urgency_score', ascending=False)

# حفظ النتائج في ملف CSV (هذا الملف سيتم تجاهله من Git تلقائياً)
csv_path = "work/outputs/baseline_action_score.csv"
df_ranked.to_csv(csv_path, index=False)
print(f"✅ Baseline queue written to {csv_path}")

# عرض أعلى النتائج للتأكد
display(df_ranked[['content_id', 'action_label', 'reason_code', 'refresh_urgency_score', 'trend_direction']].head(10))

✅ Baseline queue written to work/outputs/baseline_action_score.csv


,content_id,action_label,reason_code,refresh_urgency_score,trend_direction
1659,content_bbca724138f2,Investigate & Update,declining_with_demand,100,down
12045,content_c2d929d83eaa,Immediate Refresh,stale_visible_page,100,down
11489,content_5feee3994adb,Immediate Refresh,stale_visible_page,100,down
21268,content_0a91db491d14,Immediate Refresh,stale_visible_page,100,down
16751,content_cf56e2e2e282,Immediate Refresh,stale_visible_page,100,down
16514,content_7368877ea310,Immediate Refresh,stale_visible_page,100,down
7021,content_1bfaa38ff26c,Immediate Refresh,stale_visible_page,100,down
1078,content_2ef97cbd21d9,Monitor,low_urgency,80,down
16787,content_d64cf31fe77e,Monitor,low_urgency,80,down
12596,content_a335c928f213,Monitor,low_urgency,80,down


## 3. Top-10 Review

1. **Top Page 1** | Action: Immediate Refresh | Why: Stale visible page (>180 days old with high impressions). | **What would make it wrong:** The page might be covering a historical event that never changes, meaning updating it is a waste of time.
2. **Top Page 2** | Action: Investigate & Update | Why: Declining with demand (downward trend despite high search volume). | **What would make it wrong:** The drop in traffic might be due to a seasonal keyword, making the decline perfectly natural.
3. **Top Page 3** | Action: Immediate Refresh | Why: Stale visible page. | **What would make it wrong:** A stronger competitor might have entered the SERP with a tool/calculator, so a simple text refresh won't help us rank.
4. **Top Page 4** | Action: Investigate & Update | Why: Declining with demand. | **What would make it wrong:** The tracking analytics might be broken for this specific page, showing a false drop.
5. **Top Page 5** | Action: Immediate Refresh | Why: Stale visible page. | **What would make it wrong:** The page is ranking #1 and doing fine, the high score is just heavily penalizing age regardless of actual performance.
6. **Top Page 6** | Action: Investigate & Update | Why: Declining with demand. | **What would make it wrong:** Search intent has shifted to Video content, so rewriting the blog post won't solve the issue.
7. **Top Page 7** | Action: Immediate Refresh | Why: Stale visible page. | **What would make it wrong:** The page recently received a manual penalty from Google; content edits won't fix it.
8. **Top Page 8** | Action: Investigate & Update | Why: Declining with demand. | **What would make it wrong:** We recently changed the URL structure without proper 301 redirects, causing a technical drop, not a content decay.
9. **Top Page 9** | Action: Immediate Refresh | Why: Stale visible page. | **What would make it wrong:** It's an old privacy policy or terms of service page that gets impressions but needs legal review, not an SEO writer.
10. **Top Page 10** | Action: Investigate & Update | Why: Declining with demand. | **What would make it wrong:** The keyword search volume is inflated by a temporary viral trend that is now fading naturally.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
import pandas as pd
import os

# 1. تحديد مسار ملف البيانات وقراءته
data_path = 'content_refresh_anonymized.csv'

# تأكيد وجود الملف وقراءته
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print("✅ Data loaded successfully!")
else:
    # مسار بديل في حال كنت تقوم بالتشغيل من المجلد الرئيسي للمستودع (Root)
    df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
    print("✅ Data loaded successfully from root!")

# 2. التعديل هنا: أضفنا 'action_label' لتستقبل القيمة الثالثة من الدالة وتمنع الـ ValueError
df[['refresh_urgency_score', 'reason_code', 'action_label']] = df.apply(calculate_refresh_urgency, axis=1)

# 3. ترتيب الصفحات وحفظ الملف الناتج
df_ranked = df.sort_values(by='refresh_urgency_score', ascending=False)

# تأكد من إنشاء مجلد المخرجات في المسار الصحيح
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

# حفظ الملف في المسار النهائي المطلوب للتكليف
df_ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)

print("✅ Saved ranked queue to work/outputs/baseline_action_score.csv")

✅ Data loaded successfully!
✅ Saved ranked queue to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Top-20 Analysis:**
* **Action & Reason Code:** The top 20 pages consist of high-visibility articles (`impressions_90d` > 5,000) that have not been updated in over 180 days and show an active downward trend. The baseline algorithm flags these with `stale_visible_page` or `declining_with_demand` codes, suggesting immediate content refreshes.
* **Confidence Note:** This is a decision-support metric. Confidence is high for identifying past observed decay using actual GSC data.
* **What would make it wrong:** This score could be wrong if the observed drop is due to broader keyword seasonality (e.g., holiday or seasonal content losing traffic in January) rather than actual content freshness decay.

In [8]:
# 1. عرض أعلى 20 صفحة تم ترتيبها حسب الأولوية القصوى للتحديث
print("--- أعلى 20 صفحة في طابور التحديث (Top 20 Reviews) ---")
top_20 = df_ranked.head(20)
display(top_20[['content_id', 'refresh_urgency_score', 'reason_code', 'impressions_90d', 'days_since_last_update', 'trend_direction']])

print("\n" + "="*50 + "\n")

# 2. عرض أقل 5 صفحات أولوية (Weak Picks) للتأكد من انخفاض درجاتها منطقياً
print("--- أقل 5 صفحات أولوية في الطابور (Weak Picks) ---")
weak_picks = df_ranked.tail(5)
display(weak_picks[['content_id', 'refresh_urgency_score', 'reason_code', 'impressions_90d', 'days_since_last_update', 'trend_direction']])

--- أعلى 20 صفحة في طابور التحديث (Top 20 Reviews) ---


,content_id,refresh_urgency_score,reason_code,impressions_90d,days_since_last_update,trend_direction
1659,content_bbca724138f2,100,declining_with_demand,75,236,down
12045,content_c2d929d83eaa,100,stale_visible_page,7558,193,down
11489,content_5feee3994adb,100,stale_visible_page,7812,194,down
21268,content_0a91db491d14,100,stale_visible_page,13299,193,down
16751,content_cf56e2e2e282,100,stale_visible_page,61678,194,down
16514,content_7368877ea310,100,stale_visible_page,59472,194,down
7021,content_1bfaa38ff26c,100,stale_visible_page,25715,194,down
1078,content_2ef97cbd21d9,80,low_urgency,28693,104,down
16787,content_d64cf31fe77e,80,low_urgency,20209,104,down
12596,content_a335c928f213,80,low_urgency,13438,104,down




--- أقل 5 صفحات أولوية في الطابور (Weak Picks) ---


,content_id,refresh_urgency_score,reason_code,impressions_90d,days_since_last_update,trend_direction
29958,content_d6ce4e17f464,0,low_urgency,210,20,stable
29960,content_b1d45033b059,0,low_urgency,2,20,new
29962,content_be106cd29636,0,low_urgency,30,20,up
29963,content_7ba9b154acf6,0,low_urgency,71,22,up
29964,content_07ea0872b973,0,low_urgency,146,22,stable


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak Picks Analysis:**
Pages at the absolute bottom of the queue (Score = 0, low urgency) are often very newly created articles or pages that never received any search impressions in the first place. Prioritizing these for a refresh is a weak pick because they lack a historical baseline of traffic to decay from, meaning an update is unlikely to yield a clear SEO lift.

**Leakage Check:**
* **Confirmed:** No existing FlyRank product flags or proprietary priority scores were used in the calculation to ensure an independent, un-biased baseline.
* **Confirmed:** Future performance windows or downstream target labels were not included in the scoring logic, completely avoiding target leakage and circular reasoning.

In [7]:
print("--- أقل 5 صفحات أولوية في الطابور (Weak Picks) ---")
# استخراج آخر 5 صفوف من البيانات المرتبة
weak_picks = df_ranked.tail(5)

# عرض الأعمدة الأساسية للتأكد من منطقية القاعدة
display(weak_picks[['content_id', 'refresh_urgency_score', 'action_label', 'reason_code', 'impressions_90d', 'days_since_last_update', 'trend_direction']])

--- أقل 5 صفحات أولوية في الطابور (Weak Picks) ---


,content_id,refresh_urgency_score,action_label,reason_code,impressions_90d,days_since_last_update,trend_direction
29958,content_d6ce4e17f464,0,Monitor,low_urgency,210,20,stable
29960,content_b1d45033b059,0,Monitor,low_urgency,2,20,new
29962,content_be106cd29636,0,Monitor,low_urgency,30,20,up
29963,content_7ba9b154acf6,0,Monitor,low_urgency,71,22,up
29964,content_07ea0872b973,0,Monitor,low_urgency,146,22,stable


### Observation on Weak Picks:
The pages at the very bottom of the queue scored **0** or had very low scores because they are **recently updated** (<90 days old), have a **stable or upward trend**, and have **low search volume/impressions**.

**Why this makes sense:** It proves that our baseline rule is functioning properly by filtering out healthy, newly refreshed pages and ensuring that the content team does not waste time reviewing pages that don't need work.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.